In [ ]:
#### use this ####

In [1]:
"""
Two-way ANOVA for ChatGPT simulated ratings (CRP + NCRP), Pipeline × LLM

- Loads 9 JSONL output files
- Runs Type-II ANOVA: metric ~ pipeline + llm + pipeline×llm
- Computes eta-squared and partial eta-squared
- Saves:
  - one CSV per persona×metric (full ANOVA table)
  - combined long summary CSV
  - combined wide summary CSV
- Prints:
  - compact per-metric summaries (optional)
  - ONE clean CRP table and ONE clean NCRP table at the end
"""

import os, json
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

# -------------------------
# CONFIG
# -------------------------
SCORE_FILES = [
    ("gemma_tabular_scores.jsonl",   "tabular",  "gemma"),
    ("gemma_network_scores.jsonl",   "network",  "gemma"),
    ("gemma_bimodal_scores.jsonl",   "bimodal",  "gemma"),

    ("deepseek_tabular_scores.jsonl","tabular",  "deepseek"),
    ("deepseek_network_scores.jsonl","network",  "deepseek"),
    ("deepseek_bimodal_scores.jsonl","bimodal",  "deepseek"),

    ("gemini_tabular_scores.jsonl",  "tabular",  "gemini"),
    ("gemini_network_scores.jsonl",  "network",  "gemini"),
    ("gemini_bimodal_scores.jsonl",  "bimodal",  "gemini"),
]

CRP_METRICS  = ["crp_UND","crp_TRU","crp_INS","crp_SAT","crp_CON","crp_CVN","crp_COM","crp_USB"]
NCRP_METRICS = ["ncrp_UND","ncrp_TRU","ncrp_INS","ncrp_SAT","ncrp_CON","ncrp_CVN","ncrp_COM","ncrp_USB"]

OUT_DIR = "./anova_simulated_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

TERMS = ["C(pipeline)", "C(llm)", "C(pipeline):C(llm)"]
TERM_PRETTY = {
    "C(pipeline)": "pipe",
    "C(llm)": "llm",
    "C(pipeline):C(llm)": "pipe×llm",
}

# set False if you only want the final CRP/NCRP tables printed
PRINT_PER_METRIC = False


# -------------------------
# UTIL
# -------------------------
def load_jsonl(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    rows = []
    with open(path, "r") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"{path}: bad JSON on line {line_num}: {e}")
    return pd.DataFrame(rows)


def p_fmt(p: float) -> str:
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return f"{p:.2e}"
    return f"{p:.4f}"


def f_fmt(x: float) -> str:
    if pd.isna(x):
        return "NA"
    return f"{x:.3f}"


def es_fmt(x: float) -> str:
    if pd.isna(x):
        return "NA"
    return f"{x:.3f}"


# -------------------------
# BUILD df_all
# -------------------------
dfs = []
for path, pipeline_expected, llm_expected in SCORE_FILES:
    df = load_jsonl(path)

    # normalize strings + sanity check
    if "pipeline" not in df.columns or "llm" not in df.columns:
        raise KeyError(f"{path} missing 'pipeline' or 'llm' keys.")

    df["pipeline"] = df["pipeline"].astype(str).str.lower()
    df["llm"] = df["llm"].astype(str).str.lower()

    bad_p = (df["pipeline"] != pipeline_expected).sum()
    bad_l = (df["llm"] != llm_expected).sum()
    if bad_p:
        print(f"⚠️ {path}: {bad_p} rows have pipeline != '{pipeline_expected}'")
    if bad_l:
        print(f"⚠️ {path}: {bad_l} rows have llm != '{llm_expected}'")

    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

print("\n====================")
print("DATA LOADED")
print("====================")
print("Loaded rows:", len(df_all))
print("Pipelines:", sorted(df_all["pipeline"].unique()))
print("LLMs:", sorted(df_all["llm"].unique()))

# quick cell counts check (same for all metrics)
cell_counts_all = df_all.groupby(["pipeline", "llm"]).size().unstack(fill_value=0)
print("\nCell counts (pipeline × llm):")
print(cell_counts_all.to_string())

# missingness snapshot
cols_check = [c for c in (CRP_METRICS + NCRP_METRICS) if c in df_all.columns]
missing_snapshot = df_all[cols_check].isna().mean().sort_values(ascending=False)
print("\nMissingness snapshot (top 12):")
print(missing_snapshot.head(12).round(3).to_string())


# -------------------------
# ANOVA CORE
# -------------------------
def run_two_way_anova(df: pd.DataFrame, metric: str, typ: int = 2):

    d = df.dropna(subset=[metric, "pipeline", "llm"]).copy()
    d[metric] = pd.to_numeric(d[metric], errors="coerce")
    d = d.dropna(subset=[metric])

    if d["pipeline"].nunique() < 2 or d["llm"].nunique() < 2:
        return None, 0

    model = smf.ols(f"{metric} ~ C(pipeline) + C(llm) + C(pipeline):C(llm)", data=d).fit()
    aov = anova_lm(model, typ=typ)

    # eta-squared
    ss_total = aov["sum_sq"].sum()
    aov["eta_sq"] = aov["sum_sq"] / ss_total if ss_total != 0 else 0.0

    # partial eta-squared
    if "Residual" in aov.index:
        ss_error = aov.loc["Residual", "sum_sq"]
        aov["partial_eta_sq"] = aov["sum_sq"] / (aov["sum_sq"] + ss_error)
    else:
        aov["partial_eta_sq"] = float("nan")

    return aov, len(d)


def maybe_print_per_metric(persona, metric, aov, n_used):
    if not PRINT_PER_METRIC:
        return
    print("\n----------------------------------------")
    print(f"{persona.upper()} | {metric} | n_used={n_used}")
    print("----------------------------------------")
    rows = []
    for term in TERMS:
        if term in aov.index:
            rows.append([
                TERM_PRETTY[term],
                f_fmt(aov.loc[term, "F"]),
                p_fmt(aov.loc[term, "PR(>F)"]),
                es_fmt(aov.loc[term, "eta_sq"]),
                es_fmt(aov.loc[term, "partial_eta_sq"]),
            ])
    out = pd.DataFrame(rows, columns=["term", "F", "p", "eta_sq", "partial_eta_sq"])
    print(out.to_string(index=False))


# -------------------------
# RUN + SAVE + COLLECT
# -------------------------
summary_rows = []

def run_block(metrics, persona_label: str):
    for metric in metrics:
        if metric not in df_all.columns:
            print(f"⚠️ Missing column: {metric} (skipping)")
            continue

        aov, n_used = run_two_way_anova(df_all, metric, typ=2)
        if aov is None:
            print(f"⚠️ {metric}: insufficient factor levels.")
            continue

        maybe_print_per_metric(persona_label, metric, aov, n_used)

        # save full aov
        out_path = os.path.join(OUT_DIR, f"anova_{persona_label}_{metric}.csv")
        aov.to_csv(out_path)

        # collect summary rows
        for term in TERMS:
            if term in aov.index:
                summary_rows.append({
                    "persona": persona_label,
                    "metric": metric,
                    "term": TERM_PRETTY[term],
                    "df": float(aov.loc[term, "df"]),
                    "F": float(aov.loc[term, "F"]),
                    "p": float(aov.loc[term, "PR(>F)"]),
                    "eta_sq": float(aov.loc[term, "eta_sq"]),
                    "partial_eta_sq": float(aov.loc[term, "partial_eta_sq"]),
                    "n_used": int(n_used),
                })

print("\n====================")
print("RUNNING ANOVAs")
print("====================")
run_block(CRP_METRICS, "crp")
run_block(NCRP_METRICS, "ncrp")

summary_df = pd.DataFrame(summary_rows)

# save long summary
summary_out = os.path.join(OUT_DIR, "anova_summary_simulated_long.csv")
summary_df.to_csv(summary_out, index=False)
print("\nSaved combined long summary:", summary_out)

# save wide summary
wide = summary_df.pivot_table(
    index=["persona", "metric"],
    columns="term",
    values=["F", "p", "eta_sq", "partial_eta_sq"],
    aggfunc="first"
)
wide.columns = [f"{stat}_{term}" for stat, term in wide.columns]
wide = wide.reset_index()

wide_out = os.path.join(OUT_DIR, "anova_summary_simulated_wide.csv")
wide.to_csv(wide_out, index=False)
print("Saved combined wide summary:", wide_out)


# -------------------------
# PRINT FINAL CLEAN TABLES
# -------------------------
def make_clean_table(persona: str) -> pd.DataFrame:
    d = summary_df[summary_df["persona"] == persona].copy()
    d["metric"] = d["metric"].str.replace(f"{persona}_", "", regex=False)

    # build one row per metric with pipe/llm/int columns
    rows = []
    for metric in d["metric"].unique():
        dd = d[d["metric"] == metric].set_index("term")

        def grab(term, col):
            return dd.loc[term, col] if term in dd.index else float("nan")

        rows.append({
            "Metric": metric,
            "F_pipe": grab("pipe", "F"),
            "p_pipe": grab("pipe", "p"),
            "pη2_pipe": grab("pipe", "partial_eta_sq"),

            "F_llm": grab("llm", "F"),
            "p_llm": grab("llm", "p"),
            "pη2_llm": grab("llm", "partial_eta_sq"),

            "F_int": grab("pipe×llm", "F"),
            "p_int": grab("pipe×llm", "p"),
            "pη2_int": grab("pipe×llm", "partial_eta_sq"),
        })

    out = pd.DataFrame(rows)

    # nice ordering
    order = ["UND","TRU","INS","SAT","CON","CVN","COM","USB"]
    out["Metric"] = pd.Categorical(out["Metric"], categories=order, ordered=True)
    out = out.sort_values("Metric").reset_index(drop=True)

    # format for terminal printing
    out_print = out.copy()
    for c in ["F_pipe","F_llm","F_int","pη2_pipe","pη2_llm","pη2_int"]:
        out_print[c] = out_print[c].map(lambda x: "NA" if pd.isna(x) else f"{x:.3f}")
    for c in ["p_pipe","p_llm","p_int"]:
        out_print[c] = out_print[c].map(lambda x: "NA" if pd.isna(x) else (f"{x:.2e}" if x < 1e-4 else f"{x:.4f}"))

    return out, out_print

crp_table_raw, crp_table_print = make_clean_table("crp")
ncrp_table_raw, ncrp_table_print = make_clean_table("ncrp")

print("\n====================")
print("FINAL CLEAN TABLE — CRP")
print("====================")
print(crp_table_print.to_string(index=False))

print("\n====================")
print("FINAL CLEAN TABLE — NCRP")
print("====================")
print(ncrp_table_print.to_string(index=False))

# also save these clean tables
crp_table_raw.to_csv(os.path.join(OUT_DIR, "anova_clean_table_crp.csv"), index=False)
ncrp_table_raw.to_csv(os.path.join(OUT_DIR, "anova_clean_table_ncrp.csv"), index=False)

print("\nSaved clean tables:")
print(" -", os.path.join(OUT_DIR, "anova_clean_table_crp.csv"))
print(" -", os.path.join(OUT_DIR, "anova_clean_table_ncrp.csv"))

print("\nDone.")

<frozen importlib._bootstrap>:228: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject



DATA LOADED
Loaded rows: 888
Pipelines: ['bimodal', 'network', 'tabular']
LLMs: ['deepseek', 'gemini', 'gemma']

Cell counts (pipeline × llm):
llm       deepseek  gemini  gemma
pipeline                         
bimodal        100      97     97
network         97      97    100
tabular        100     100    100

Missingness snapshot (top 12):
crp_UND     0.0
crp_TRU     0.0
crp_INS     0.0
crp_SAT     0.0
crp_CON     0.0
crp_CVN     0.0
crp_COM     0.0
crp_USB     0.0
ncrp_UND    0.0
ncrp_TRU    0.0
ncrp_INS    0.0
ncrp_SAT    0.0

RUNNING ANOVAs

Saved combined long summary: ./anova_simulated_outputs/anova_summary_simulated_long.csv
Saved combined wide summary: ./anova_simulated_outputs/anova_summary_simulated_wide.csv

FINAL CLEAN TABLE — CRP
Metric  F_pipe    p_pipe pη2_pipe  F_llm    p_llm pη2_llm  F_int    p_int pη2_int
   UND 471.115 8.97e-140    0.517 51.290 8.55e-22   0.105 54.234 6.95e-41   0.198
   TRU 288.654  4.30e-97    0.396  0.575   0.5628   0.001 52.728 7.66e-40   0.19